<a href="https://colab.research.google.com/github/oumaimabelgaied/GNN_fraud_detection/blob/main/Fraud_Detection_GRAPHSAGE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#GraphSAGE

Ce notebook implemente le modèle GraphSAGE pour la détéction d'anomalies / fraudes financières sur le Dataset Elliptic++ .

GraphSAGE est une méthode non spectrale utilisée pour "neighborhood sampling" .
C'est un modèle GNN inductif, ce qui le distingue fondamentalement de modèles comme GCN. Expliquation :

Au lieu d'apprendre des embeddings fixes pour chaque nœud (transductif), GraphSAGE apprend une fonction d'agrégation qui peut généraliser à des nœuds jamais vus pendant l'entraînement — essentiel pour la détection de fraude où de nouvelles transactions arrivent en continu.

On utilisera aussi Max Pooling comme fonction d'aggrégation car il détecte des signaux extrêmes dans le voisinage : un seul voisin très suspect suffit à faire ressortir un nœud frauduleux avec l'inconvénient de la compléxité élevée . MLP sur chaque voisin puis max element-wise sur les résultats.


In [4]:
from google.colab import drive
drive.mount('/content/drive')
folder_path = "/content/drive/MyDrive/RT4/RT4(ouma)/PFA/Elliptic_Dataset"

Mounted at /content/drive


In [ ]:
import os
os.chdir(folder_path)

print(os.getcwd())

In [ ]:
import pandas as pd
txs = pd.read_csv("/content/drive/MyDrive/RT4/RT4(ouma)/PFA/Elliptic_Dataset/AddrTx_edgelist.csv")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from scipy import stats
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (15, 8)
plt.rcParams['font.size'] = 10

##Chargement des données

In [ ]:
# Charger les données (adapter les chemins selon votre structure)
try:
    # Features des transactions
    features = pd.read_csv('/content/drive/MyDrive/RT4/RT4(ouma)/PFA/Elliptic_Dataset/txs_features.csv')
    print(f"✓ Features chargées: {features.shape}")

    # Classes des transactions
    classes = pd.read_csv('/content/drive/MyDrive/RT4/RT4(ouma)/PFA/Elliptic_Dataset/txs_classes.csv')
    print(f"✓ Classes chargées: {classes.shape}")

    # Edges (flux de Bitcoins)
    edges = pd.read_csv('/content/drive/MyDrive/RT4/RT4(ouma)/PFA/Elliptic_Dataset/txs_edgelist.csv')
    print(f"✓ Edges chargées: {edges.shape}")

    # Données optionnelles (wallets)
    try:
        addr_addr = pd.read_csv('/content/drive/MyDrive/RT4/RT4(ouma)/PFA/Elliptic_Dataset/AddrAddr_edgelist.csv')
        addr_tx = pd.read_csv('/content/drive/MyDrive/RT4/RT4(ouma)/PFA/Elliptic_Dataset/AddrTx_edgelist.csv')
        tx_addr = pd.read_csv('/content/drive/MyDrive/RT4/RT4(ouma)/PFA/Elliptic_Dataset/TxAddr_edgelist.csv')
        print(f"✓ Données wallets chargées")
        wallet_data_available = True
    except:
        print("⚠ Données wallets non disponibles (optionnel)")
        wallet_data_available = False

except FileNotFoundError as e:
    print(f"❌ Erreur: {e}")
    print("⚠ Veuillez placer les fichiers CSV dans le répertoire courant")
    exit(1)

In [ ]:
print(features.head())

In [ ]:
print(classes.head())

##Nettoyage des données

valeurs manquantes (imputation médiane) / outliers (IQR clipping) / unicité (txId)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

features = pd.read_csv('/content/drive/MyDrive/RT4/RT4(ouma)/PFA/Elliptic_Dataset/txs_features.csv')

# ── Renommer les colonnes ──────────────────────────
# Col 0 = txId, col 1 = time_step, cols 2..183 = features
features.columns = ['txId', 'time_step'] + [f'f{i}' for i in range(182)]
print(features.head())

# ── Valeurs manquantes ────────────────────────────
nan_counts = features.isnull().sum()
nan_total  = nan_counts.sum()
print(f"\n[NaN] Total valeurs manquantes : {nan_total}")
if nan_total > 0:
    print(nan_counts[nan_counts > 0])
    # Imputation par médiane (robuste aux outliers)
    from sklearn.impute import SimpleImputer
    feat_cols = [f'f{i}' for i in range(182)]
    imp = SimpleImputer(strategy='median')
    features[feat_cols] = imp.fit_transform(features[feat_cols])
    print("  → Imputation médiane appliquée")
else:
    print("  → Aucun NaN détecté, dataset propre")

# ── Types et cohérence ────────────────────────────
assert features['txId'].nunique() == len(features), \
    "ERREUR : txId non unique !"
print(f"\n[IDs] txId uniques : {features['txId'].nunique()} — OK")
print(f"[TIME] Time steps présents : {sorted(features['time_step'].unique())[:5]} ... "
      f"{sorted(features['time_step'].unique())[-3:]}")

# ── Mapping classes ───────────────────────────────
class_map = {1.0: 1, 2.0: 0, 3.0: -1} #3.0 correspond à unknown
classes['label'] = classes['class'].map(class_map)

n_illicit = (classes['label'] == 1).sum()
n_licit   = (classes['label'] == 0).sum()
n_unknown = (classes['label'] == -1).sum()

print(f"\n[CLASSES]")
print(f"  Illicite  (1) : {n_illicit:>7,}  ({100*n_illicit/(n_illicit+n_licit):.1f}% des labellisés)")
print(f"  Licite    (0) : {n_licit:>7,}  ({100*n_licit/(n_illicit+n_licit):.1f}% des labellisés)")
print(f"  Unknown  (-1) : {n_unknown:>7,}")
print(f"  Ratio licit/illicit : {n_licit/n_illicit:.1f}:1")

# ── Outliers — IQR clipping sur features locales ─
feat_cols = [f'f{i}' for i in range(182)]
q1  = features[feat_cols].quantile(0.01)
q99 = features[feat_cols].quantile(0.99)
features[feat_cols] = features[feat_cols].clip(lower=q1, upper=q99, axis=1)
print(f"\n[OUTLIERS] Clipping 1%-99% appliqué sur {len(feat_cols)} features")

# ── Fusion master dataframe ───────────────────────
df = features.merge(classes[['txId', 'label']], on='txId', how='left')
df['label'] = df['label'].fillna(-1).astype(int)

print(f"\n[MERGE] Dataframe final : {df.shape}")
print(f"  → {df.shape[1] - 3} features + txId + time_step + label")
print("\n✓ Nettoyage terminé")
print(df.head())

##EDA & Corrélation

In [ ]:
feat_cols = [f'f{i}' for i in range(182)]
df_labeled = df[df['label'] != -1].copy()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('EDA — Elliptic++ Dataset', fontsize=14, fontweight='bold')

# ── Distribution des classes ──────────────────────
ax = axes[0, 0]
vals   = [n_licit, n_illicit]
labels = ['Licite (0)', 'Illicite (1)']
colors = ['#1D9E75', '#E24B4A']
bars = ax.bar(labels, vals, color=colors, edgecolor='white', linewidth=1.5)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 300,
            f'{v:,}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_title('Distribution des classes labellisées', fontsize=12)
ax.set_ylabel('Nombre de transactions')
ax.spines[['top','right']].set_visible(False)

# ── Transactions par time step ────────────────────
ax = axes[0, 1]
ts_counts = df.groupby(['time_step', 'label']).size().unstack(fill_value=0)
ts_counts.rename(columns={-1:'Unknown', 0:'Licite', 1:'Illicite'}, inplace=True)
ts_counts[['Licite','Illicite']].plot(kind='bar', ax=ax,
    color=['#1D9E75', '#E24B4A'], edgecolor='none', alpha=0.85)
ax.set_title('Transactions labellisées par time step', fontsize=12)
ax.set_xlabel('Time step')
ax.set_ylabel('Nombre')
ax.tick_params(axis='x', rotation=45, labelsize=7)
ax.spines[['top','right']].set_visible(False)

# ── Matrice de corrélation (top 30 features) ─────
ax = axes[0, 2]
# Sélectionner les 30 features les plus discriminantes par variance inter-classe
means_ill = df_labeled[df_labeled['label']==1][feat_cols].mean()
means_lic = df_labeled[df_labeled['label']==0][feat_cols].mean()
diff = (means_ill - means_lic).abs()
top30 = diff.nlargest(30).index.tolist()

corr_matrix = df_labeled[top30].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, ax=ax, cmap='RdBu_r',
            vmin=-1, vmax=1, center=0,
            xticklabels=False, yticklabels=False,
            cbar_kws={'shrink': 0.8})
ax.set_title('Corrélation — top 30 features discriminantes', fontsize=12)

# ── Distribution de 4 features clés (boxplot) ────
ax = axes[1, 0]
# Features f0..f3 sont les features locales les plus informatives (Elliptic++)
feat_sample = top30[:4]
plot_data = []
for f in feat_sample:
    for label, name in [(0, 'Licite'), (1, 'Illicite')]:
        vals_f = df_labeled[df_labeled['label']==label][f].values
        for v in vals_f:
            plot_data.append({'feature': f, 'classe': name, 'valeur': v})
plot_df = pd.DataFrame(plot_data)
import seaborn as sns
sns.boxplot(data=plot_df, x='feature', y='valeur', hue='classe',
            palette={'Licite':'#1D9E75','Illicite':'#E24B4A'},
            ax=ax, flierprops=dict(markersize=1, alpha=0.3))
ax.set_title('Top 4 features — distribution par classe', fontsize=12)
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=15)
ax.spines[['top','right']].set_visible(False)

# ── Degré des nœuds ───────────────────────────────
ax = axes[1, 1]
from collections import Counter
all_nodes = pd.concat([edges['txId1'], edges['txId2']])
degree_counts = Counter(all_nodes)
degrees = list(degree_counts.values())
ax.hist(degrees, bins=80, color='#534AB7', edgecolor='none', alpha=0.85,
        log=True)
ax.axvline(np.mean(degrees), color='#E24B4A', lw=2,
           label=f'Moyenne = {np.mean(degrees):.1f}')
ax.set_title('Distribution des degrés (log scale)', fontsize=12)
ax.set_xlabel('Degré du nœud')
ax.set_ylabel('Fréquence (log)')
ax.legend(fontsize=10)
ax.spines[['top','right']].set_visible(False)

# ── Homophilie visuelle ───────────────────────────
ax = axes[1, 2]
homophily_data = {
    'ill→ill': 998,
    'lic→lic': 33930,
    'ill→lic': 915,
    'lic→ill': 781,
}
h_colors = ['#E24B4A','#1D9E75','#EF9F27','#EF9F27']
bars = ax.bar(homophily_data.keys(), homophily_data.values(),
              color=h_colors, edgecolor='white', linewidth=1.5)
for bar, v in zip(bars, homophily_data.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{v:,}', ha='center', va='bottom', fontsize=9)
ax.set_title(f'Homophilie du graphe (ratio=0.954)', fontsize=12)
ax.set_ylabel("Nombre d'edges")
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.show()

# ── Rapport de multicolinéarité ──────────────────
corr_full = df_labeled[feat_cols].corr().abs()
np.fill_diagonal(corr_full.values, 0)

high_corr_pairs = (corr_full > 0.90).sum().sum() // 2
med_corr_pairs  = ((corr_full > 0.70) & (corr_full <= 0.90)).sum().sum() // 2

print(f"\n[MULTICOLINÉARITÉ]")
print(f"  Paires avec |r| > 0.90 : {high_corr_pairs}")
print(f"  Paires avec |r| > 0.70 : {med_corr_pairs}")
print(f"  → PCA fortement recommandée")

##Analyse en composantes principale : RÉDUCTION DIMENSIONNELLE

justifiée par la forte multicolinéarité --> on réduit donc la dimmensionnalité avec PCA ( ou feature selection des caractéristiques discriminantes )

justification avec référence provenant de la littérature :
*  Weber et al. (2019) : des statistiques sur les features locales → redondance structurelle forte
* Elmougy & Liu (2023) : idem sur Elliptic++. PCA est le choix standard pour
gérer la multicolinéarité avant un GNN.

In [ ]:
feat_cols = [f'f{i}' for i in range(182)]

# ── Normalisation AVANT PCA ───────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[feat_cols].values)
print(f"[NORM] Features normalisées : mean≈0, std≈1 — shape {X_scaled.shape}")

# ── PCA exploratoire (variance expliquée) ─────────
pca_full = PCA(n_components=min(182, len(df)-1), random_state=42)
pca_full.fit(X_scaled)

cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_90   = np.searchsorted(cumvar, 0.90) + 1
n_95   = np.searchsorted(cumvar, 0.95) + 1
n_99   = np.searchsorted(cumvar, 0.99) + 1

print(f"\n[PCA] Variance expliquée cumulée :")
print(f"  {n_90:>3} composantes → 90% de variance")
print(f"  {n_95:>3} composantes → 95% de variance")
print(f"  {n_99:>3} composantes → 99% de variance")

# ── Choix : 95% de variance → N_COMPONENTS ───────
# Décision : 95% est le consensus dans la littérature
# pour les datasets tabulaires avant GNN
# (voir aussi : Pareja et al. 2020 EvolveGCN)
N_COMPONENTS = n_95
print(f"\n[DÉCISION] N_COMPONENTS = {N_COMPONENTS} (95% variance)")

# ── Visualisation scree plot ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(range(1, 61), pca_full.explained_variance_ratio_[:60] * 100,
        'o-', color='#534AB7', markersize=4, linewidth=1.5)
ax.set_title('Scree plot — variance par composante', fontsize=12)
ax.set_xlabel('Composante')
ax.set_ylabel('Variance expliquée (%)')
ax.spines[['top','right']].set_visible(False)

ax = axes[1]
ax.plot(range(1, 183), cumvar * 100, color='#1D9E75', linewidth=2)
ax.axhline(90, color='#EF9F27', lw=1.5, ls='--', label='90%')
ax.axhline(95, color='#E24B4A', lw=1.5, ls='--', label='95%')
ax.axvline(N_COMPONENTS, color='#E24B4A', lw=1.5, ls=':')
ax.fill_between(range(1, N_COMPONENTS+1),
                cumvar[:N_COMPONENTS] * 100, alpha=0.15, color='#1D9E75')
ax.set_title(f'Variance cumulée — seuil choisi : {N_COMPONENTS} composantes', fontsize=12)
ax.set_xlabel('Nombre de composantes')
ax.set_ylabel('Variance expliquée cumulée (%)')
ax.legend(fontsize=10)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.show()

# ── Application PCA finale ────────────────────────
pca = PCA(n_components=N_COMPONENTS, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Nommer les composantes
pca_cols = [f'pc{i}' for i in range(N_COMPONENTS)]
df_pca = pd.DataFrame(X_pca, columns=pca_cols)
df_pca['txId']      = df['txId'].values
df_pca['time_step'] = df['time_step'].values
df_pca['label']     = df['label'].values

print(f"\n[PCA] Matrice réduite : {df_pca.shape}")
print(f"  {182} features → {N_COMPONENTS} composantes principales")
print(f"  Variance conservée : {cumvar[N_COMPONENTS-1]*100:.2f}%")

# ── Vérification séparabilité dans l'espace PCA ──
df_lab = df_pca[df_pca['label'] != -1]
fig, ax = plt.subplots(figsize=(8, 6))
for label, name, color in [(0,'Licite','#1D9E75'), (1,'Illicite','#E24B4A')]:
    sub = df_lab[df_lab['label']==label]
    ax.scatter(sub['pc0'], sub['pc1'], c=color, label=name,
               alpha=0.3, s=4, edgecolors='none')
ax.set_title('Projection PCA — PC1 vs PC2 (nœuds labellisés)', fontsize=12)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.legend(markerscale=4, fontsize=11)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

print("\n✓ PCA terminée — df_pca prêt pour construction du graphe")

##Construction du graphe PyG

In [ ]:
import torch
from torch_geometric.data import Data

# ── Mapping txId → index entier ──────────────────
id_to_idx = {txid: idx for idx, txid in enumerate(df_pca['txId'].values)}
N = len(id_to_idx)
print(f"[GRAPHE] Nombre de nœuds : {N:,}")

# ── Tensor des features (composantes PCA) ─────────
pca_cols = [f'pc{i}' for i in range(N_COMPONENTS)]
x = torch.tensor(df_pca[pca_cols].values, dtype=torch.float)
print(f"[FEATURES] x shape : {x.shape}  (N × {N_COMPONENTS} composantes PCA)")

# ── Tensor des edges ──────────────────────────────
valid = edges['txId1'].isin(id_to_idx) & edges['txId2'].isin(id_to_idx)
e = edges[valid]
src = e['txId1'].map(id_to_idx).values
dst = e['txId2'].map(id_to_idx).values

# Bidirectionnel : GraphSAGE agrège dans les deux sens
src_bi = np.concatenate([src, dst])
dst_bi = np.concatenate([dst, src])
edge_index = torch.tensor(np.array([src_bi, dst_bi]), dtype=torch.long)
print(f"[EDGES] edge_index shape : {edge_index.shape}  ({edge_index.shape[1]:,} arêtes bidirectionnelles)")

# ── Labels ────────────────────────────────────────
y = torch.tensor(df_pca['label'].values, dtype=torch.long)
time = df_pca['time_step'].values

# ── Masques temporels (recommandation EDA ) ────
# Train : t ∈ [1, 40]   Val : t ∈ [41, 45]   Test : t ∈ [46, 49]
labeled = (df_pca['label'].values != -1)

train_mask = torch.tensor(labeled & (time <= 40), dtype=torch.bool)
val_mask   = torch.tensor(labeled & (time >= 41) & (time <= 45), dtype=torch.bool)
test_mask  = torch.tensor(labeled & (time >= 46), dtype=torch.bool)

print(f"\n[MASQUES TEMPORELS]")
print(f"  Train  (t≤40)    : {train_mask.sum():>6,} nœuds labellisés")
print(f"  Val    (t 41-45) : {val_mask.sum():>6,} nœuds labellisés")
print(f"  Test   (t 46-49) : {test_mask.sum():>6,} nœuds labellisés")

# Vérification : pas de fuite temporelle
assert y[test_mask].shape[0] > 0, "Test set vide !"

# ── Poids de classe ───────────────────────────────
y_train = y[train_mask].numpy()
n0 = (y_train == 0).sum()   # licite
n1 = (y_train == 1).sum()   # illicite
# Inverse frequency : pénalise davantage les erreurs sur illicite
w0 = n1 / (n0 + n1)
w1 = n0 / (n0 + n1)
class_weights = torch.tensor([w0, w1], dtype=torch.float)
print(f"\n[POIDS] Classe licite   : {w0:.4f}")
print(f"[POIDS] Classe illicite : {w1:.4f}  (×{w1/w0:.1f} boosting)")

# ── Objet Data PyG ───────────────────────────────
data = Data(
    x          = x,
    edge_index = edge_index,
    y          = y,
    train_mask = train_mask,
    val_mask   = val_mask,
    test_mask  = test_mask,
)
data.time_step     = torch.tensor(time, dtype=torch.long)
data.class_weights = class_weights

print(f"\n{'='*45}")
print("Objet Data PyG :")
print(data)
print(f"  Dirigé         : {data.is_directed()}")
print(f"  Isolés         : {data.has_isolated_nodes()}")
print(f"  Self-loops     : {data.has_self_loops()}")
print(f"{'='*45}")
print("\n✓ Graphe construit — prêt pour GraphSAGE")

##Modèle GraphSAGE

In [ ]:
!pip install torch-geometric

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv

class GraphSAGE(nn.Module):
    """
    2-layer GraphSAGE avec max aggregation.
    Référence : Hamilton et al. (2017), agrégateur
    max pooling recommandé pour la détection de fraude
    (capture les signaux extrêmes dans le voisinage).
    """
    def __init__(self, in_channels, hidden_channels,
                 out_channels, dropout=0.5):
        super().__init__()
        # Couche 1 : voisinage 1-hop
        self.conv1 = SAGEConv(in_channels, hidden_channels,
                              aggr='max')
        # Couche 2 : voisinage 2-hop
        self.conv2 = SAGEConv(hidden_channels, hidden_channels,
                              aggr='max')
        # Batch normalization (stabilise l'entraînement)
        self.bn1 = nn.BatchNorm1d(hidden_channels)
        self.bn2 = nn.BatchNorm1d(hidden_channels)
        # Classificateur
        self.classifier = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels // 2, out_channels),
        )
        self.dropout = dropout

    def forward(self, x, edge_index):
        # Couche 1
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        # Couche 2
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        # Classification
        return self.classifier(x)

    def get_embeddings(self, x, edge_index):
        """Retourne les embeddings avant classification."""
        with torch.no_grad():
            x = F.relu(self.bn1(self.conv1(x, edge_index)))
            x = F.relu(self.bn2(self.conv2(x, edge_index)))
        return x


# ── Instanciation ─────────────────────────────────────
HIDDEN  = 256
DROPOUT = 0.4

model = GraphSAGE(
    in_channels     = 56,   # nb composantes PCA
    hidden_channels = HIDDEN,
    out_channels    = 2,              # binaire : licite / illicite
    dropout         = DROPOUT,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = model.to(device)
data   = data.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"[MODÈLE] GraphSAGE — {total_params:,} paramètres")
print(f"[DEVICE] {device}")
print(model)

##Entraînement

In [ ]:
from sklearn.metrics import f1_score, average_precision_score
import torch.optim as optim

# ── Hyperparamètres ───────────────────────────────────
LR         = 3e-4
WEIGHT_DECAY = 1e-4
EPOCHS     = 300
PATIENCE   = 30        # early stopping

optimizer  = optim.AdamW(model.parameters(),
                         lr=LR, weight_decay=WEIGHT_DECAY)
scheduler  = optim.lr_scheduler.ReduceLROnPlateau(
                 optimizer, mode='max', factor=0.5,
                 patience=10)

cw = data.class_weights.to(device)
criterion = nn.CrossEntropyLoss(weight=cw)

# ── Boucle d'entraînement ────────────────────────────
history = {'train_loss': [], 'val_loss': [],
           'val_f1': [], 'val_auprc': []}

best_val_auprc = 0.0
best_epoch     = 0
patience_counter = 0

print(f"Entraînement — {EPOCHS} époques max, early stopping patience={PATIENCE}\n")

for epoch in range(1, EPOCHS + 1):

    # ── TRAIN ──────────────────────────────────────────
    model.train()
    optimizer.zero_grad()
    out  = model(data.x, data.edge_index)
    loss = criterion(out[data.train_mask],
                     data.y[data.train_mask])
    loss.backward()
    # Gradient clipping (stabilité sur graphes denses)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    # ── VALIDATION ─────────────────────────────────────
    model.eval()
    with torch.no_grad():
        out_val = model(data.x, data.edge_index)
        val_loss = criterion(out_val[data.val_mask],
                             data.y[data.val_mask]).item()

        probs = F.softmax(out_val, dim=1)[:, 1].cpu().numpy()
        preds = out_val.argmax(dim=1).cpu().numpy()
        y_val = data.y[data.val_mask].cpu().numpy()

        val_f1    = f1_score(y_val, preds[data.val_mask.cpu()],
                             zero_division=0)
        val_auprc = average_precision_score(
                        y_val,
                        probs[data.val_mask.cpu()])

    history['train_loss'].append(loss.item())
    history['val_loss'].append(val_loss)
    history['val_f1'].append(val_f1)
    history['val_auprc'].append(val_auprc)

    scheduler.step(val_auprc)

    # ── EARLY STOPPING sur AUPRC ───────────────────────
    if val_auprc > best_val_auprc:
        best_val_auprc   = val_auprc
        best_epoch       = epoch
        patience_counter = 0
        torch.save(model.state_dict(), '/content/drive/My Drive/RT4/RT4(ouma)/PFA/best_graphsage.pt')
    else:
        patience_counter += 1

    if epoch % 20 == 0 or epoch == 1:
        print(f"Ep {epoch:>3} | train_loss={loss.item():.4f} "
              f"| val_loss={val_loss:.4f} "
              f"| val_F1={val_f1:.3f} "
              f"| val_AUPRC={val_auprc:.3f}")

    if patience_counter >= PATIENCE:
        print(f"\n→ Early stopping à l'époque {epoch} "
              f"(meilleur AUPRC={best_val_auprc:.4f} @ ep {best_epoch})")
        break

# ── Courbes d'apprentissage ───────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(history['train_loss'], label='Train', color='#534AB7')
axes[0].plot(history['val_loss'],   label='Val',   color='#E24B4A')
axes[0].set_title('Loss')
axes[0].legend(); axes[0].spines[['top','right']].set_visible(False)

axes[1].plot(history['val_f1'], color='#1D9E75')
axes[1].set_title('Val F1-score (illicite)')
axes[1].spines[['top','right']].set_visible(False)

axes[2].plot(history['val_auprc'], color='#D85A30')
axes[2].set_title('Val AUPRC (métrique principale)')
axes[2].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('/content/drive/My Drive/RT4/RT4(ouma)/PFA/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\n✓ Meilleur modèle sauvegardé → best_graphsage.pt")

##Evaluation

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score,
    precision_recall_curve, roc_curve, f1_score
)

# ── Charger le meilleur modèle ────────────────────────
model.load_state_dict(torch.load('/content/drive/My Drive/RT4/RT4(ouma)/PFA/best_graphsage.pt',map_location=device))
model.eval()

with torch.no_grad():
    out   = model(data.x, data.edge_index)
    probs = F.softmax(out, dim=1)[:, 1].cpu().numpy()
    preds = out.argmax(dim=1).cpu().numpy()

y_test  = data.y[data.test_mask].cpu().numpy()
p_test  = probs[data.test_mask.cpu().numpy()]
pr_test = preds[data.test_mask.cpu().numpy()]

# ── Métriques principales ────────────────────────────
auprc  = average_precision_score(y_test, p_test)
auroc  = roc_auc_score(y_test, p_test)
f1_ill = f1_score(y_test, pr_test, pos_label=1, zero_division=0)
f1_lic = f1_score(y_test, pr_test, pos_label=0, zero_division=0)

print("=" * 50)
print("RÉSULTATS SUR LE TEST SET (t ∈ [46, 49])")
print("=" * 50)
print(f"  AUPRC       : {auprc:.4f}   ← métrique principale")
print(f"  AUC-ROC     : {auroc:.4f}")
print(f"  F1 illicite : {f1_ill:.4f}")
print(f"  F1 licite   : {f1_lic:.4f}")
print()
print(classification_report(y_test, pr_test,
      target_names=['Licite', 'Illicite']))

# ── Figures d'évaluation ─────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Matrice de confusion
cm = confusion_matrix(y_test, pr_test)
sns.heatmap(cm, annot=True, fmt='d', ax=axes[0],
            cmap='Blues', cbar=False,
            xticklabels=['Prédit Licite', 'Prédit Illicite'],
            yticklabels=['Vrai Licite',  'Vrai Illicite'])
axes[0].set_title(f'Matrice de confusion\n(Test t 46-49)', fontsize=12)

# Courbe PR
prec, rec, _ = precision_recall_curve(y_test, p_test)
axes[1].plot(rec, prec, color='#D85A30', lw=2,
             label=f'AUPRC = {auprc:.3f}')
axes[1].axhline(y_test.mean(), ls='--', color='gray',
                label=f'Baseline = {y_test.mean():.3f}')
axes[1].set_xlabel('Rappel')
axes[1].set_ylabel('Précision')
axes[1].set_title('Courbe Precision-Recall', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].spines[['top','right']].set_visible(False)

# Courbe ROC
fpr, tpr, _ = roc_curve(y_test, p_test)
axes[2].plot(fpr, tpr, color='#534AB7', lw=2,
             label=f'AUC = {auroc:.3f}')
axes[2].plot([0,1],[0,1], 'k--', lw=1)
axes[2].set_xlabel('Taux faux positifs')
axes[2].set_ylabel('Taux vrais positifs')
axes[2].set_title('Courbe ROC', fontsize=12)
axes[2].legend(fontsize=10)
axes[2].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('/content/drive/My Drive/RT4/RT4(ouma)/PFA/evaluation_results.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Analyse par time step ─────────────────────────────
print("\n[ANALYSE PAR TIME STEP — TEST]")
print(f"{'Step':>5} | {'N illicite':>10} | {'F1':>6} | {'AUPRC':>7}")
print("-" * 38)

time_test = data.time_step[data.test_mask].cpu().numpy()
for t in sorted(np.unique(time_test)):
    mask_t = (time_test == t)
    if mask_t.sum() == 0 or y_test[mask_t].sum() == 0:
        continue
    f1_t    = f1_score(y_test[mask_t], pr_test[mask_t],
                       pos_label=1, zero_division=0)
    auprc_t = average_precision_score(y_test[mask_t], p_test[mask_t])
    print(f"  t={t:>2} | {y_test[mask_t].sum():>10} | {f1_t:>6.3f} | {auprc_t:>7.3f}")

print("\n✓ Évaluation terminée")